# Comparación local con SRBench 2025 en Google Colab

Ejecuta el perfil `official` del **harness local de WarpSymbolic**: 24 datasets × 30 semillas = **720 tareas**. Esto sirve para diagnóstico interno y una comparación exploratoria con resultados publicados. **No ejecuta `experiment/analyze.py` de SRBench, no reproduce su ajuste de hiperparámetros ni constituye una evaluación aceptada por los creadores.** El JSONL declara `official_protocol=false`.

Antes de comenzar, selecciona **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**. El perfil fija **3600 s como techo por tarea**, 50.000 individuos y 150 generaciones. El runner y la búsqueda adaptativa respetan ahora el techo de tiempo; una tarea puede terminar antes al alcanzar otro criterio de parada. Colab puede interrumpir una sesión; vuelve a ejecutar las celdas desde el principio para reanudar. Cada tarea terminada queda guardada en Google Drive.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
RUN_ROOT = Path('/content/drive/MyDrive/WarpSymbolic/SRBench2025')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
REF_FILE = RUN_ROOT / 'commit.txt'
REPO = Path('/content/WarpSymbolic')
REPO_URL = 'https://github.com/juansito17/Algoritmo-Genetico---Formulas.git'
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
if REF_FILE.exists():
    pinned_commit = REF_FILE.read_text(encoding='utf-8').strip()
    current_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    if current_commit != pinned_commit:
        subprocess.run(['git', 'fetch', '--depth', '1', 'origin', pinned_commit], cwd=REPO, check=True)
        subprocess.run(['git', 'checkout', '--detach', pinned_commit], cwd=REPO, check=True)
else:
    pinned_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    REF_FILE.write_text(pinned_commit + '\n', encoding='utf-8')
OUTPUT = RUN_ROOT / pinned_commit[:12] / 'official.jsonl'
CACHE = RUN_ROOT / 'cache'
RANKING = OUTPUT.with_name('ranking.json')
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print('Commit fijado:', pinned_commit)
print('Resultados:', OUTPUT)

## Instalar y verificar CUDA

La compilación nativa tarda varios minutos al iniciar una sesión nueva de Colab. Se detiene si no hay GPU CUDA o si la extensión no carga.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO) + '[benchmark]'], check=True)
runner_source = (REPO / 'src/warpsymbolic/cli/srbench_runner.py').read_text(encoding='utf-8')
adaptive_source = (REPO / 'src/AlphaSymbolic/experimental/adaptive_search.py').read_text(encoding='utf-8')
if 'min(float(context.fit_time_limit_sec), 60.0)' in runner_source or 'min(float(estimator.max_time), 60.0)' in adaptive_source:
    raise RuntimeError('El commit fijado aún recorta la búsqueda a 60 s. Publica la corrección y usa un RUN_ROOT nuevo antes de medir.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Activa una GPU CUDA en la configuración de Colab.')
device = torch.cuda.get_device_properties(0)
print(f'GPU: {device.name}; VRAM: {device.total_memory / 2**30:.1f} GiB; PyTorch: {torch.__version__}')
build_env = os.environ.copy()
build_env['TORCH_CUDA_ARCH_LIST'] = f'{device.major}.{device.minor}'
build_env['MAX_JOBS'] = '2'
cuda_dir = REPO / 'src' / 'warpsymbolic' / 'gpu' / 'cuda'
if not list(cuda_dir.glob('rpn_cuda_native*.so')):
    subprocess.run([sys.executable, 'setup.py', 'build_ext', '--inplace'], cwd=cuda_dir, env=build_env, check=True)
from warpsymbolic.gpu.cuda_loader import load_rpn_cuda_native
print('Extensión CUDA:', load_rpn_cuda_native().__file__)
subprocess.run(['nvidia-smi'], check=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
environment_file = OUTPUT.parent / f'environment_{stamp}.txt'
with environment_file.open('w', encoding='utf-8') as record:
    record.write(f'commit={pinned_commit}\npython={sys.version}\ntorch={torch.__version__}\ntorch_cuda={torch.version.cuda}\n')
    for args in (['nvidia-smi'], ['nvcc', '--version'], [sys.executable, '-m', 'pip', 'freeze']):
        record.write('\n$ ' + ' '.join(args) + '\n')
        record.write(subprocess.run(args, capture_output=True, text=True, check=True).stdout)
print('Entorno guardado:', environment_file)

## Verificar el plan

Esta celda no entrena: confirma la cobertura y los presupuestos configurados. Verifica que el commit clonado de WarpSymbolic incluya la eliminación del antiguo recorte interno de 60 s.

In [ ]:
BASE = [sys.executable, '-u', '-m', 'warpsymbolic.cli.benchmark_srbench', '--profile', 'official', '--cache-dir', str(CACHE), '--output', str(OUTPUT), '--resume']
subprocess.run(BASE + ['--dry-run'], cwd=REPO, check=True)

## Preparar datos y referencias

Descarga y comprueba hashes de los 24 datasets y de los resultados oficiales usados para el ranking. El caché queda en Drive para las sesiones siguientes.

In [ ]:
subprocess.run(BASE + ['--prepare-only', '--rank'], cwd=REPO, check=True)

## Ejecutar las 720 tareas de la comparación local

La salida aparece en vivo por dataset y semilla. Los registros se escriben y sincronizan en Drive al terminar cada tarea. Si Colab corta la sesión, repite las celdas: `--resume` salta los registros ya guardados. Los errores previos también se saltan; para reintentarlos añade `--retry-failed` al comando.

In [ ]:
run_env = os.environ.copy()
run_env['PYTHONUNBUFFERED'] = '1'
result = subprocess.run(BASE, cwd=REPO, env=run_env)
print('Código de salida:', result.returncode, '| JSONL:', OUTPUT)
if result.returncode:
    print('Revisa los registros con status=error antes de generar el ranking final.')

## Comprobar cobertura y comparar

Genera el ranking solo cuando los 720 pares dataset/semilla tengan un registro exitoso. El archivo `ranking.json` queda junto al JSONL en Drive.

In [ ]:
from warpsymbolic.cli.benchmark_srbench import load_manifest, read_jsonl, DEFAULT_MANIFEST
manifest = load_manifest(DEFAULT_MANIFEST)
expected = {(item['name'], int(seed)) for item in manifest['datasets'] for seed in manifest['seeds']}
latest = {}
for row in read_jsonl(OUTPUT):
    if row.get('record_type') == 'srbench_run':
        latest[(row['dataset'], int(row['seed']))] = row
successful = {key for key, row in latest.items() if row.get('status') == 'ok'}
failed = {key for key, row in latest.items() if row.get('status') != 'ok'}
print(f'Completadas: {len(successful & expected)}/{len(expected)}; errores: {len(failed & expected)}; pendientes: {len(expected - successful - failed)}')
if successful == expected and not failed:
    subprocess.run(BASE + ['--rank-only', '--ranking-output', str(RANKING)], cwd=REPO, check=True)
    ranking = json.loads(RANKING.read_text(encoding='utf-8'))
    print('Ranking:', RANKING)
    print('Comparable con protocolo oficial:', ranking['comparable_to_official'])
else:
    print('Ejecuta de nuevo la celda de benchmark para continuar; usa --retry-failed si hubo errores.')

## Antes de solicitar una evaluación de SRBench

Esta comparación local no reemplaza las pruebas de los creadores. El repositorio contiene `integrations/srbench/prepare_upstream.sh` para copiar el método a un checkout fijado de SRBench y `integrations/srbench/run_upstream_24x30.sh` para usar `experiment/analyze.py`. Antes de pedir revisión hay que publicar y fijar un commit de WarpSymbolic, construir y probar su imagen en un entorno Linux con CUDA, ejecutar los tests upstream y conservar sus resultados brutos y el registro de hardware. La aceptación depende de la revisión de los mantenedores.